# Notebook 05: Efficiency Benchmarking

## Objective
This notebook focuses on benchmarking route performance to identify the most efficient and least efficient logistics routes.

### Key Tasks:
- Rank routes based on average lead time
- Identify top 10 most efficient routes
- Identify bottom 10 least efficient routes
- Develop a Route Efficiency Score
- Compare performance across ship modes

This step enables data-driven decision-making for logistics optimization.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
# Load route-level dataset
route_state = pd.read_csv("../data/route_state_summary.csv")

route_state.head()

,Route_State,Total_Shipments,Avg_Lead_Time,Lead_Time_Std,Min_Lead_Time,Max_Lead_Time
0,Lot's O' Nuts → Alabama,34,4.59,1.84,1,7
1,Lot's O' Nuts → Alberta,16,3.50,1.32,2,7
2,Lot's O' Nuts → Arizona,111,3.86,2.03,0,7
3,Lot's O' Nuts → Arkansas,31,4.19,1.54,0,7
4,Lot's O' Nuts → British Columbia,18,5.00,1.53,2,7


In [3]:
# Rank routes (lower lead time = better)
route_state["Efficiency_Rank"] = route_state["Avg_Lead_Time"].rank(method="dense")

In [4]:
route_state.sort_values("Efficiency_Rank").head()

,Route_State,Total_Shipments,Avg_Lead_Time,Lead_Time_Std,Min_Lead_Time,Max_Lead_Time,Efficiency_Rank
129,The Other Factory → North Carolina,1,0.0,NaN,0,0,1.0
120,The Other Factory → Kentucky,2,1.0,1.41,0,2,2.0
131,The Other Factory → Oregon,2,1.5,0.71,1,2,3.0
54,Lot's O' Nuts → West Virginia,2,1.5,2.12,0,3,3.0
112,The Other Factory → Arizona,2,2.0,0.00,2,2,4.0


## Top 10 Most Efficient Routes
Routes with lowest average lead time.

In [5]:
top_10_routes = route_state.sort_values("Avg_Lead_Time").head(10)

top_10_routes

,Route_State,Total_Shipments,Avg_Lead_Time,Lead_Time_Std,Min_Lead_Time,Max_Lead_Time,Efficiency_Rank
129,The Other Factory → North Carolina,1,0.00,NaN,0,0,1.0
120,The Other Factory → Kentucky,2,1.00,1.41,0,2,2.0
131,The Other Factory → Oregon,2,1.50,0.71,1,2,3.0
54,Lot's O' Nuts → West Virginia,2,1.50,2.12,0,3,3.0
112,The Other Factory → Arizona,2,2.00,0.00,2,2,4.0
125,The Other Factory → Nevada,1,2.00,NaN,2,2,4.0
126,The Other Factory → New Hampshire,2,2.00,0.00,2,2,4.0
108,Sugar Shack → Tennessee,1,2.00,NaN,2,2,4.0
80,Secret Factory → New Mexico,2,2.00,1.41,1,3,4.0
62,Secret Factory → Colorado,7,2.14,1.35,0,4,5.0


## Bottom 10 Least Efficient Routes
Routes with highest average lead time.

In [6]:
bottom_10_routes = route_state.sort_values("Avg_Lead_Time", ascending=False).head(10)

bottom_10_routes

,Route_State,Total_Shipments,Avg_Lead_Time,Lead_Time_Std,Min_Lead_Time,Max_Lead_Time,Efficiency_Rank
133,The Other Factory → Rhode Island,1,7.00,NaN,7,7,96.0
124,The Other Factory → Missouri,1,7.00,NaN,7,7,96.0
127,The Other Factory → New Jersey,3,6.33,1.15,5,7,95.0
71,Secret Factory → Maryland,3,6.00,1.00,5,7,94.0
75,Secret Factory → Mississippi,1,6.00,NaN,6,6,94.0
99,Sugar Shack → Delaware,1,6.00,NaN,6,6,94.0
107,Sugar Shack → Rhode Island,1,6.00,NaN,6,6,94.0
78,Secret Factory → New Hampshire,1,6.00,NaN,6,6,94.0
138,The Other Factory → Washington,4,5.75,0.96,5,7,93.0
35,Lot's O' Nuts → Newfoundland And Labrador,6,5.67,1.63,3,7,92.0


## Route Efficiency Score

We normalize average lead time into a score where:

- Higher score = better efficiency
- Lower score = worse efficiency

In [7]:
# Normalize lead time (min-max scaling)
min_lt = route_state["Avg_Lead_Time"].min()
max_lt = route_state["Avg_Lead_Time"].max()

route_state["Efficiency_Score"] = (
    (max_lt - route_state["Avg_Lead_Time"]) / (max_lt - min_lt)
) * 100

route_state["Efficiency_Score"] = route_state["Efficiency_Score"].round(2)

In [8]:
route_state.sort_values("Efficiency_Score", ascending=False).head()

,Route_State,Total_Shipments,Avg_Lead_Time,Lead_Time_Std,Min_Lead_Time,Max_Lead_Time,Efficiency_Rank,Efficiency_Score
129,The Other Factory → North Carolina,1,0.0,NaN,0,0,1.0,100.00
120,The Other Factory → Kentucky,2,1.0,1.41,0,2,2.0,85.71
54,Lot's O' Nuts → West Virginia,2,1.5,2.12,0,3,3.0,78.57
131,The Other Factory → Oregon,2,1.5,0.71,1,2,3.0,78.57
125,The Other Factory → Nevada,1,2.0,NaN,2,2,4.0,71.43


In [9]:
#Performance Segmentation
# Categorize routes
route_state["Performance_Category"] = pd.cut(
    route_state["Efficiency_Score"],
    bins=[0, 40, 70, 100],
    labels=["Poor", "Average", "High"]
)

route_state.head()

,Route_State,Total_Shipments,Avg_Lead_Time,Lead_Time_Std,Min_Lead_Time,Max_Lead_Time,Efficiency_Rank,Efficiency_Score,Performance_Category
0,Lot's O' Nuts → Alabama,34,4.59,1.84,1,7,79.0,34.43,Poor
1,Lot's O' Nuts → Alberta,16,3.50,1.32,2,7,21.0,50.00,Average
2,Lot's O' Nuts → Arizona,111,3.86,2.03,0,7,38.0,44.86,Average
3,Lot's O' Nuts → Arkansas,31,4.19,1.54,0,7,60.0,40.14,Average
4,Lot's O' Nuts → British Columbia,18,5.00,1.53,2,7,86.0,28.57,Poor


## Ship Mode Performance Comparison

In [10]:
# Load featured dataset
ncd = pd.read_csv("../data/featured_nassau_data.csv")

# Compare lead time by ship mode
ship_mode_perf = ncd.groupby("Ship Mode").agg(
    Avg_Lead_Time=("Shipping Lead Time", "mean"),
    Total_Shipments=("Order ID", "count")
).reset_index()

ship_mode_perf["Avg_Lead_Time"] = ship_mode_perf["Avg_Lead_Time"].round(2)

ship_mode_perf

,Ship Mode,Avg_Lead_Time,Total_Shipments
0,First Class,2.02,1548
1,Same Day,0.50,547
2,Second Class,3.48,1979
3,Standard Class,5.00,6120


In [11]:
route_state.head()

,Route_State,Total_Shipments,Avg_Lead_Time,Lead_Time_Std,Min_Lead_Time,Max_Lead_Time,Efficiency_Rank,Efficiency_Score,Performance_Category
0,Lot's O' Nuts → Alabama,34,4.59,1.84,1,7,79.0,34.43,Poor
1,Lot's O' Nuts → Alberta,16,3.50,1.32,2,7,21.0,50.00,Average
2,Lot's O' Nuts → Arizona,111,3.86,2.03,0,7,38.0,44.86,Average
3,Lot's O' Nuts → Arkansas,31,4.19,1.54,0,7,60.0,40.14,Average
4,Lot's O' Nuts → British Columbia,18,5.00,1.53,2,7,86.0,28.57,Poor


In [12]:
route_state.to_csv("../data/route_efficiency_benchmark.csv", index=False)
top_10_routes.to_csv("../data/top_10_routes.csv", index=False)
bottom_10_routes.to_csv("../data/bottom_10_routes.csv", index=False)
ship_mode_perf.to_csv("../data/ship_mode_performance.csv", index=False)

print("Benchmarking datasets saved successfully.")

Benchmarking datasets saved successfully.
